## Custom RNN in raw python using numpy to understand maths behind the RNN Algorithm

In [ ]:
!python -m spacy download en_core_web_lg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.7/587.7 MB 3.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
!pip install PyMuPDF

In [ ]:
import fitz

In [ ]:
def convert_pdf_to_text(pdf_path, output_path):
  doc = fitz.open(pdf_path)
  text = ""

  for page_num in range(len(doc)):
    page = doc.load_page(page_num)
    text += page.get_text()

  with open(output_path, 'w', encoding='utf-8') as text_file:
    text_file.write(text)

In [ ]:
pdf_path = 'War and Peace.pdf'
output_path = 'war_and_peace.txt'

In [ ]:
convert_pdf_to_text(pdf_path, output_path)

In [ ]:
import spacy
import numpy as np
import random

In [ ]:
nlp = spacy.load("en_core_web_lg")
nlp.max_length = 15000000000

In [ ]:
with open(output_path, 'r', encoding='utf-8') as file:
  text = file.read()

In [ ]:
text

'War and Peace \n \n3 of 2882 \nChapter I \n‘Well, Prince, so Genoa and Lucca are now just family \nestates of the Buonapartes. But I warn you, if you don’t \ntell me that this means war, if you still try to defend the \ninfamies and horrors perpetrated by that Antichrist - I \nreally believe he is Antichrist - I will have nothing more \nto do with you and you are no longer my friend, no longer \nmy ‘faithful slave,’ as you call yourself! But how do you \ndo? I see I have frightened you - sit down and tell me all \nthe news.’ \nIt was in July, 1805, and the speaker was the well-\nknown Anna Pavlovna Scherer, maid of honor and \nfavorite of the Empress Marya Fedorovna. With these \nwords she greeted Prince Vasili Kuragin, a man of high \nrank and importance, who was the first to arrive at her \nreception. Anna Pavlovna had had a cough for some days. \nShe was, as she said, suffering from la grippe; grippe \nbeing then a new word in St. Petersburg, used only by the \nelite. \nAll her inv

In [ ]:
doc = nlp(text.lower())

In [ ]:
words = [token.text for token in doc if not token.is_punct]

In [ ]:
vocab = list(set(words))

In [ ]:
len(vocab)

21738

In [ ]:
word_to_idx = {word: i for i, word in enumerate(vocab)}
idx_to_word = {i: word for i, word in enumerate(vocab)}

In [ ]:
sequence_length = 5
sequences = []
next_words = []

In [ ]:
for i in range(len(words) - sequence_length):
  sequences.append(words[i:i + sequence_length])
  next_words.append(words[i + sequence_length])


In [ ]:
X = np.array([[word_to_idx[word] for word in seq] for seq in sequences])
y = np.array([word_to_idx[word] for word in next_words])

In [ ]:
class SimpleRNN:
  def __init__(self, vocab_size, hidden_size, output_size):
    #initialize weights randomly
    self.Wxh = np.random.randn(hidden_size, vocab_size) * 0.01
    self.Whh = np.random.randn(hidden_size, hidden_size) * 0.01  # hidden to hidden
    self.Why = np.random.randn(output_size, hidden_size) * 0.01  # hidden to output
    self.bh = np.zeros((hidden_size, 1))  # hidden bias
    self.by = np.zeros((output_size, 1))  # output bias

  def forward(self, inputs, h_prev):
    xs, hs, ys, ps = {}, {}, {}, {}
    h_prev = np.zeros((300, 1))
    hs[-1] = np.copy(h_prev)

    for t in range(len(inputs)):
      xs[t] = np.zeros((vocab_size, 1))
      xs[t][inputs[t]] = 1
      # Hidden state (memory)
      hs[t] = np.tanh(np.dot(self.Wxh, xs[t]) + np.dot(self.Whh, hs[t-1]) + self.bh)
      # Output (unnormalized scores)
      ys[t] = np.dot(self.Why, hs[t]) + self.by

      # Softmax to get probabilities
      ps[t] = np.exp(ys[t]) / np.sum(np.exp(ys[t]))

    return xs, hs, ps

  def loss(self, ps, targets):
    return -np.sum(np.log(ps[t][targets[t], 0]) for t in range(len(targets)))

  def backward(self, xs, hs, ps, targets):
        # Initialize gradients
        dWxh, dWhh, dWhy = np.zeros_like(self.Wxh), np.zeros_like(self.Whh), np.zeros_like(self.Why)
        dbh, dby = np.zeros_like(self.bh), np.zeros_like(self.by)
        dh_next = np.zeros_like(hs[0])

        for t in reversed(range(len(targets))):
            dy = np.copy(ps[t])
            dy[targets[t]] -= 1  # Backpropagate into y

            dWhy += np.dot(dy, hs[t].T)
            dby += dy
            dh = np.dot(self.Why.T, dy) + dh_next
            dhraw = (1 - hs[t] * hs[t]) * dh  # Backprop through tanh
            dbh += dhraw
            dWxh += np.dot(dhraw, xs[t].T)
            dWhh += np.dot(dhraw, hs[t-1].T)
            dh_next = np.dot(self.Whh.T, dhraw)

        for dparam in [dWxh, dWhh, dWhy, dbh, dby]:
            np.clip(dparam, -5, 5, out=dparam)  # Clip to prevent exploding gradients

        return dWxh, dWhh, dWhy, dbh, dby




In [ ]:
vocab_size = len(vocab)
hidden_size = 300
seq_length = 5
learning_rate = 0.1

In [ ]:
rnn = SimpleRNN(vocab_size, hidden_size, vocab_size)

In [ ]:
h_prev = np.zeros(hidden_size - 1)
n = len(X)

In [ ]:
X.shape

(659901, 5)

In [ ]:
for epoch in range(4000):  # number of epochs
    # Select random sample
    idx = random.randint(0, n - seq_length - 1)
    inputs = X[idx]
    targets = y[idx:idx+seq_length]

    # Forward pass
    xs, hs, ps = rnn.forward(inputs, h_prev)

    # Compute loss
    loss = rnn.loss(ps, targets)

    # Backward pass
    dWxh, dWhh, dWhy, dbh, dby = rnn.backward(xs, hs, ps, targets)

    # Update weights using gradient descent
    rnn.Wxh -= learning_rate * dWxh
    rnn.Whh -= learning_rate * dWhh
    rnn.Why -= learning_rate * dWhy
    rnn.bh -= learning_rate * dbh
    rnn.by -= learning_rate * dby

    # Print loss at each epoch
    if epoch % 100 == 0:
        print(f'Epoch {epoch}, Loss: {loss}')

<ipython-input-82-fc68c8a7284a>:29: DeprecationWarning: Calling np.sum(generator) is deprecated, and in the future will give a different result. Use np.sum(np.fromiter(generator)) or the python sum builtin instead.
  return -np.sum(np.log(ps[t][targets[t], 0]) for t in range(len(targets)))


Epoch 0, Loss: 49.93910470118907
Epoch 100, Loss: 33.238364211830294
Epoch 200, Loss: 41.67965243935738
Epoch 300, Loss: 28.88139410495326
Epoch 400, Loss: 42.712277394181505
Epoch 500, Loss: 35.25890201723662
Epoch 600, Loss: 32.101114479504844
Epoch 700, Loss: 34.08998783655501
Epoch 800, Loss: 32.69895521857329
Epoch 900, Loss: 29.642882556533245
Epoch 1000, Loss: 30.616342546368905
Epoch 1100, Loss: 22.749232312622183
Epoch 1200, Loss: 33.10262044498364
Epoch 1300, Loss: 41.91399531461121
Epoch 1400, Loss: 22.230083632503078
Epoch 1500, Loss: 29.55962310671142
Epoch 1600, Loss: 34.64986571514214
Epoch 1700, Loss: 29.43350786890237
Epoch 1800, Loss: 33.47562014919094
Epoch 1900, Loss: 23.34690876024093
Epoch 2000, Loss: 36.01218829837717
Epoch 2100, Loss: 32.080094821063746
Epoch 2200, Loss: 36.14483586604061
Epoch 2300, Loss: 39.50724030080491
Epoch 2400, Loss: 24.187057421425376
Epoch 2500, Loss: 30.157541467883973
Epoch 2600, Loss: 32.88046999492226
Epoch 2700, Loss: 22.110496353

In [ ]:
def predict(rnn, start_sequence, word_to_idx, idx_to_word, n_predictions=1):
    h_prev = np.zeros((hidden_size, 1))  # Reset hidden state
    inputs = [word_to_idx[word] for word in start_sequence]

    # Forward pass
    xs, hs, ps = rnn.forward(inputs, h_prev)

    # Predict the next word
    last_word_probs = ps[len(start_sequence) - 1]
    predicted_word_idx = np.argmax(last_word_probs)

    # Convert index back to word
    return idx_to_word[predicted_word_idx]

In [ ]:
# Example of sentence completion
start_sequence = ["vicomte", "merely", "shrugged", "his", "shoulders"]
predicted_word = predict(rnn, start_sequence, word_to_idx, idx_to_word)
print(" ".join(start_sequence) + " " + predicted_word)

vicomte merely shrugged his shoulders and
